Zad1

In [2]:
import sqlite3
import requests

print("Pobieranie danych z API...")
response = requests.get("https://randomuser.me/api/?results=30")
users = response.json()["results"]

conn = sqlite3.connect("users.db")
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS Users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name TEXT,
    last_name TEXT,
    email TEXT,
    age INTEGER,
    gender TEXT,
    country TEXT
)
''')

cursor.execute("DELETE FROM Users")

insert_query = '''
INSERT INTO Users (first_name, last_name, email, age, gender, country)
VALUES (?, ?, ?, ?, ?, ?)
'''

print("Zapisywanie danych do bazy SQLite...")
for user in users:
    first_name = user['name']['first']
    last_name = user['name']['last']
    email = user['email']
    age = user['dob']['age']
    gender = user['gender']
    country = user['location']['country']

    cursor.execute(insert_query, (first_name, last_name, email, age, gender, country))

conn.commit()

print("\n--- WYNIKI ANALIZY ---")

cursor.execute("SELECT gender, COUNT(*) FROM Users GROUP BY gender")
print("\n1. Podział na płeć:")
for row in cursor.fetchall():
    print(f" - {row[0]}: {row[1]}")

cursor.execute("SELECT AVG(age) FROM Users")
avg_age = cursor.fetchone()[0]
print(f"\n2. Średni wiek użytkowników: {avg_age:.1f} lat")

cursor.execute("SELECT COUNT(DISTINCT country) FROM Users")
countries_count = cursor.fetchone()[0]
print(f"\n3. Użytkownicy mieszkają łącznie w {countries_count} krajach.")

cursor.execute("SELECT country, COUNT(*) FROM Users GROUP BY country ORDER BY COUNT(*) DESC")
print("\nRozkład na poszczególne kraje:")
for row in cursor.fetchall():
    print(f" - {row[0]}: {row[1]}")

conn.close()

Pobieranie danych z API...
Zapisywanie danych do bazy SQLite...

--- WYNIKI ANALIZY ---

1. Podział na płeć:
 - female: 17
 - male: 13

2. Średni wiek użytkowników: 52.0 lat

3. Użytkownicy mieszkają łącznie w 17 krajach.

Rozkład na poszczególne kraje:
 - United Kingdom: 3
 - Spain: 3
 - Mexico: 3
 - United States: 2
 - New Zealand: 2
 - Netherlands: 2
 - Iran: 2
 - India: 2
 - Finland: 2
 - Denmark: 2
 - Ukraine: 1
 - Turkey: 1
 - Switzerland: 1
 - Norway: 1
 - Germany: 1
 - France: 1
 - Canada: 1


Zad2

In [3]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 23.1 MB/s eta 0:00:00


In [9]:
from pymongo import MongoClient
import requests

client = MongoClient("mongodb://localhost:27017")
db = client.lab4
networks = db["networks"]

networks.drop()

print("Pobieranie danych z GeckoTerminal API...")
response = requests.get("https://api.geckoterminal.com/api/v2/networks")

if response.status_code == 200:
    data = response.json()["data"]

    print(f"Zapisywanie {len(data)} sieci do bazy...")
    networks.insert_many(data)

    pipeline = [
        {"$group": {
            "_id": "$type",
            "count": {"$sum": 1}
        }},
        {"$sort": {"count": -1}}
    ]

    print("\n--- WYNIKI AGREGACJI ---")
    print("Liczba sieci według typu:")

    for doc in networks.aggregate(pipeline):
        print(f" - Typ '{doc['_id']}': {doc['count']} szt.")

else:
    print(f"Błąd podczas łączenia z API: kod {response.status_code}")
client.close()

KeyboardInterrupt: 

Zad3

In [10]:
import numpy as np

# "Baza" filmow z embeddingami (w prawdziwym systemie np. z OpenAI API)
filmy = {
    "Incepcja":          np.array([0.8, 0.3, 0.9]),
    "Matrix":            np.array([0.75, 0.35, 0.85]),
    "Toy Story":         np.array([0.2, 0.9, 0.1]),
    "Shrek":             np.array([0.25, 0.85, 0.15]),
    "Szeregowiec Ryan":  np.array([0.6, 0.1, 0.7]),
}

def semantic_search(query_vec, database, top_k=3):
    results = []

    query_norm = np.linalg.norm(query_vec)

    for title, doc_vec in database.items():
        dot_product = np.dot(query_vec, doc_vec)

        doc_norm = np.linalg.norm(doc_vec)

        if query_norm == 0 or doc_norm == 0:
            sim = 0.0
        else:
            sim = dot_product / (query_norm * doc_norm)

        results.append((title, sim))

    results.sort(key=lambda x: x[1], reverse=True)

    return results[:top_k]

if __name__ == "__main__":
    query = np.array([0.7, 0.3, 0.8])

    print("Szukam filmów najbardziej podobnych do wektora zapytania...\n")
    results = semantic_search(query, filmy, top_k=3)

    print("Wyniki:")
    for title, sim in results:
        print(f"  {title}: {sim:.3f}")

Szukam filmów najbardziej podobnych do wektora zapytania...

Wyniki:
  Matrix: 1.000
  Incepcja: 0.999
  Szeregowiec Ryan: 0.986
